## Modelado



## 1. Importación de Librerías y carga de datos

In [4]:
!wget https://raw.githubusercontent.com/FIJI-1919/REPOSITORIO_TRABAJO_GRUPAL_EV1/refs/heads/main/data/dataset_clientes.csv


--2026-05-25 18:18:32--  https://raw.githubusercontent.com/FIJI-1919/REPOSITORIO_TRABAJO_GRUPAL_EV1/refs/heads/main/data/dataset_clientes.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3282843 (3.1M) [text/plain]
Saving to: ‘dataset_clientes.csv’

dataset_clientes.cs 100%[===================>]   3.13M  14.8MB/s    in 0.2s    

2026-05-25 18:18:33 (14.8 MB/s) - ‘dataset_clientes.csv’ saved [3282843/3282843]



In [3]:
import os
import io
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

In [7]:
data_original = pd.read_csv('dataset_clientes.csv')

## 3. Clases y Funciones de Preprocesamiento

Definimos aquí todo el código de limpieza y transformación:
- **`Winsorizer`**: recorta valores extremos (atípicos) al percentil 5% y 95%.
- **`eliminar_duplicados`**: elimina filas con `id_cliente` repetido.
- **`tratar_negativos`**: convierte negativos en variables financieras a `NaN`.
- **`crear_features`**: genera `ratio_deuda` y variables cíclicas de hora.
- **`build_preprocessor`**: construye el `ColumnTransformer` con pipelines numérico y categórico.
- **`preprocess_pipeline`**: función principal que ejecuta todo en secuencia.

In [8]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Recorta los valores extremos de cada columna numérica al percentil
    inferior y superior indicado en `limits`.

    Parámetros
    ----------
    limits : tuple, default (0.05, 0.05)
        Fracción a recortar en cada cola (inferior, superior).
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        self.columns_ = X.columns if isinstance(X, pd.DataFrame) else np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X[col] = np.clip(X[col].astype("float64"), lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columns_ if input_features is None else input_features)


# ── Funciones de limpieza ──
def eliminar_duplicados(df: pd.DataFrame) -> pd.DataFrame:
    """Elimina registros duplicados basándose en 'id_cliente'."""
    antes = len(df)
    df = df.drop_duplicates(subset=["id_cliente"]).copy()
    print(f"  Duplicados eliminados: {antes - len(df)}")
    return df


def tratar_negativos(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte a NaN los valores negativos en variables financieras."""
    columnas = ["ingreso_mensual", "gasto_mensual", "deuda_total"]
    df = df.copy()
    for col in columnas:
        n = (df[col] < 0).sum()
        if n > 0:
            print(f"  '{col}': {n} valores negativos convertidos a NaN")
        df.loc[df[col] < 0, col] = np.nan
    return df


def crear_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera nuevas variables:
      - ratio_deuda : deuda_total / (ingreso_mensual + 1)
      - hora_sin    : seno cíclico de hora_registro (24 h)
      - hora_cos    : coseno cíclico de hora_registro (24 h)
    """
    df = df.copy()
    df["ratio_deuda"] = df["deuda_total"] / (df["ingreso_mensual"] + 1)
    df["hora_sin"]    = np.sin(2 * np.pi * df["hora_registro"] / 24)
    df["hora_cos"]    = np.cos(2 * np.pi * df["hora_registro"] / 24)
    return df


print("Clases y funciones de preprocesamiento definidas.")

Clases y funciones de preprocesamiento definidas.


In [13]:
# ── Definición de columnas ──
COLS_EXCLUIR       = ["id_cliente", "fecha_registro", "codigo_postal"]
NUMERIC_FEATURES   = ["edad", "ingreso_mensual", "antiguedad_meses", "frecuencia_compra",
                       "ultima_compra_dias", "num_productos", "hora_registro",
                       "gasto_mensual", "deuda_total", "score_crediticio",
                       "ratio_deuda", "hora_sin", "hora_cos"]
CATEGORICAL_FEATURES = ["genero", "estado_civil", "canal_registro", "dia_semana_registro",
                         "region", "tipo_plan", "uso_app"]
BINARY_FEATURES    = ["tiene_tarjeta_credito"]


def build_preprocessor() -> ColumnTransformer:
    """
    Construye el ColumnTransformer:
      - Numérico  : Winsorizer → SimpleImputer(mean) → StandardScaler
      - Categórico: SimpleImputer(most_frequent) → OneHotEncoder
    """
    numeric_transformer = Pipeline(steps=[
        ("winsorizer", Winsorizer()),
        ("imputer",    SimpleImputer(strategy="mean")),
        ("escalado",   StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES + BINARY_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ])


def preprocess_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pipeline completo:
      1. Eliminar duplicados por id_cliente.
      2. Tratar negativos → NaN.
      3. Crear features derivadas.
      4. Aplicar ColumnTransformer (Winsorizer + Imputer + Scaler + OHE).
    """
    print("Iniciando preprocesamiento...")
    df = eliminar_duplicados(df)
    df = tratar_negativos(df)
    df = crear_features(df)

    y     = df["abandono"].reset_index(drop=True)
    X_raw = df.drop(columns=[c for c in COLS_EXCLUIR + ["abandono"] if c in df.columns])

    preprocessor = build_preprocessor()
    X_array      = preprocessor.fit_transform(X_raw)

    ohe_cols = (preprocessor
                .named_transformers_["cat"]
                .named_steps["onehot"]
                .get_feature_names_out(CATEGORICAL_FEATURES)
                .tolist())
    all_cols = NUMERIC_FEATURES + BINARY_FEATURES + ohe_cols

    df_clean            = pd.DataFrame(X_array, columns=all_cols)
    df_clean["abandono"] = y.values

    print(f"Preprocesamiento finalizado. Shape: {df_clean.shape}")
    return df_clean


print("Pipeline de preprocesamiento listo.")

Pipeline de preprocesamiento listo.


## 4. Definición de Modelos

In [14]:
def get_classification_models() -> dict:
    """
    Retorna pipelines base de clasificación:
      - LogisticRegression     : modelo lineal interpretable, baseline estadístico.
      - DecisionTreeClassifier : captura no linealidades, fácil de interpretar.
      - SVM                    : robusto en alta dimensión, buen margen de separación.
    """
    return {
        "LogisticRegression": Pipeline(steps=[
            ("classifier", LogisticRegression(
                max_iter=1000, random_state=42, class_weight="balanced"
            )),
        ]),
        "DecisionTreeClassifier": Pipeline(steps=[
            ("classifier", DecisionTreeClassifier(
                random_state=42, class_weight="balanced"
            )),
        ]),
        "SVM": Pipeline(steps=[
            ("classifier", SVC(
                kernel="rbf", probability=True,
                random_state=42, class_weight="balanced"
            )),
        ]),
    }


def get_regression_models() -> dict:
    """
    Retorna pipelines base de regresión:
      - LinearRegression      : baseline simple, alta interpretabilidad.
      - DecisionTreeRegressor : captura relaciones no lineales en el gasto.
    """
    return {
        "LinearRegression": Pipeline(steps=[
            ("regressor", LinearRegression()),
        ]),
        "DecisionTreeRegressor": Pipeline(steps=[
            ("regressor", DecisionTreeRegressor(random_state=42)),
        ]),
    }


def train_and_fit_model(pipeline, X_train, y_train):
    """Entrena un pipeline y lo retorna ajustado."""
    pipeline.fit(X_train, y_train)
    return pipeline


print("Modelos definidos correctamente.")

Modelos definidos correctamente.


## 5. Aplicamos Preprocesamiento

In [15]:
data_limpio = preprocess_pipeline(data_original)

print(f"\nDimensiones finales : {data_limpio.shape}")
print(f"Nulos remanentes    : {data_limpio.isnull().sum().sum()}")
display(data_limpio.head())

Iniciando preprocesamiento...
  Duplicados eliminados: 400
  'ingreso_mensual': 11 valores negativos convertidos a NaN
  'gasto_mensual': 66 valores negativos convertidos a NaN
  'deuda_total': 137 valores negativos convertidos a NaN
Preprocesamiento finalizado. Shape: (20000, 33)

Dimensiones finales : (20000, 33)
Nulos remanentes    : 0


,edad,ingreso_mensual,antiguedad_meses,frecuencia_compra,ultima_compra_dias,num_productos,hora_registro,gasto_mensual,deuda_total,score_crediticio,...,dia_semana_registro_Miercoles,dia_semana_registro_Sabado,dia_semana_registro_Viernes,region_Norte,region_Sur,tipo_plan_Estandar,tipo_plan_Premium,uso_app_Bajo,uso_app_Medio,abandono
0,0.995847,0.529309,1.213020,-0.724598,1.589155,0.002765,1.565456,0.928263,0.566241,-1.616594,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1
1,0.141534,1.884570,0.142729,-0.358339,1.202924,0.711810,-0.210486,-0.648098,-0.556150,-0.283686,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0
2,-0.029329,0.000000,-1.581629,0.374180,0.478739,0.711810,-0.802467,-0.100184,1.866782,1.852509,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0
3,0.312397,-1.637991,-1.581629,-1.273987,-0.168199,-0.706279,0.677485,0.126219,1.313808,-1.757904,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1
4,-0.997550,-0.286424,-1.373516,-0.907728,0.971185,0.002765,-0.506476,0.679392,-0.532902,-1.474195,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1


In [16]:
# Guardar dataset limpio para los notebooks siguientes
os.makedirs('data', exist_ok=True)
data_limpio.to_csv('data/dataset_clientes_limpio.csv', index=False)
print("Dataset limpio guardado en: 'data/dataset_clientes_limpio.csv'")

Dataset limpio guardado en: 'data/dataset_clientes_limpio.csv'


In [19]:
from google.colab import files
data_limpio.to_csv('dataset_clientes_limpio.csv', index=False)
files.download('dataset_clientes_limpio.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. División Train / Test

In [23]:
#target: `abandono` cladificacion

SEMILLA = 42

X_clf = data_limpio.drop(columns=['abandono'])
y_clf = data_limpio['abandono']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEMILLA, stratify=y_clf
)

print("=== CLASIFICACIÓN ===")
print(f"Train : {X_train_clf.shape[0]} filas | Test : {X_test_clf.shape[0]} filas")
print(f"Tasa abandono train : {y_train_clf.mean():.2%}")
print(f"Tasa abandono test  : {y_test_clf.mean():.2%}")

=== CLASIFICACIÓN ===
Train : 16000 filas | Test : 4000 filas
Tasa abandono train : 39.67%
Tasa abandono test  : 39.67%


In [25]:
#target: `gasto_mensual` regresión

X_reg = data_limpio.drop(columns=['gasto_mensual', 'abandono'])
y_reg = data_limpio['gasto_mensual']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=SEMILLA
)

print("=== REGRESIÓN ===")
print(f"Train : {X_train_reg.shape[0]} filas | Test : {X_test_reg.shape[0]} filas")

=== REGRESIÓN ===
Train : 16000 filas | Test : 4000 filas


## 7. Entrenamiento de Modelos Base

## Clasificadores

In [ ]:
clasificadores_base = {}

print("Entrenando clasificadores base...")
for nombre, pipeline in get_classification_models().items():
    print(f"  -> {nombre}...")
    clasificadores_base[nombre] = train_and_fit_model(pipeline, X_train_clf, y_train_clf)

print("\nClasificadores entrenados:", list(clasificadores_base.keys()))

Entrenando clasificadores base...
  -> LogisticRegression...
  -> DecisionTreeClassifier...
  -> SVM...


## Regresores

In [ ]:
regresores_base = {}

print("Entrenando regresores base...")
for nombre, pipeline in get_regression_models().items():
    print(f"  -> {nombre}...")
    regresores_base[nombre] = train_and_fit_model(pipeline, X_train_reg, y_train_reg)

print("\nRegresores entrenados:", list(regresores_base.keys()))

Score en Entrenamiento

In [ ]:
print("=== SCORE EN ENTRENAMIENTO (clasificadores) ===")
for nombre, modelo in clasificadores_base.items():
    score = modelo.score(X_train_clf, y_train_clf)
    print(f"  {nombre:<30}: {score:.4f}")

print("\n=== SCORE EN ENTRENAMIENTO (regresores) ===")
for nombre, modelo in regresores_base.items():
    score = modelo.score(X_train_reg, y_train_reg)
    print(f"  {nombre:<30}: {score:.4f}")